In [ ]:
import sys
sys.path.append('../src')
from should_be_stdlib import resample_log
from neurodata import *
from data import *

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import Akima1DInterpolator

In [ ]:
#### ½ octave with equal temperament (reasonable guess)
old_freqs = resample_log([3, 3*2**4], 9)
new_freqs = resample_log([3, 3*2**4], 16)
old_freqs, new_freqs

In [ ]:
# rescale tuning curves
def normalize(x, c=1):
    """Normalize vector and rescale to c"""
    # Multiply then divide for numerical stability
    return x * c / np.linalg.norm(x, ord=1)

In [ ]:
record = load_set()
tuning_curves = get_tc(record)
coords = get_xyz(record)

In [ ]:
tuning_curves.columns = old_freqs.round(1)
plt.plot(tuning_curves.T, alpha=0.5)
plt.title('Neural tuning curves')
plt.xlabel('Stimulus Frequency (kHz)')
plt.ylabel('Response ($ΔF/F_0$)')
plt.xscale('log', base=2)
plt.tight_layout()
plt.savefig(figspath('tuning-curves.png'), dpi=300)
plt.show()

In [ ]:
# tuning curves can't be negative. scale to pi so values range from [0, pi]
tuning_curves_rescaled = tuning_curves.apply(
    lambda x: normalize(x, c=np.pi),
    axis=1,
    result_type='expand',
)
tuning_curves_rescaled.to_csv(datapath('data_tuning-curves_rescaled.csv'))
plt.plot(tuning_curves_rescaled.T, alpha=0.5)
plt.title('Neural tuning curves (rescaled)')
plt.xlabel('Stimulus Frequency (kHz)')
plt.ylabel('Scaled response ($ΔF/F_0$)')
plt.xscale('log', base=2)
plt.tight_layout()
plt.savefig(figspath('tuning-curves-rescaled.png'), dpi=300)
plt.show()

In [ ]:
# upsample the tuning curves to fill the binary space
tuning_curves_resampled = tuning_curves.apply(
    lambda response: Akima1DInterpolator(old_freqs, response, method='makima')(new_freqs),
    axis=1,
    result_type='expand',
)
tuning_curves_resampled.columns = new_freqs.round(1)
tuning_curves_resampled.to_csv(datapath('data_tuning-curves_resampled.csv'))
plt.plot(tuning_curves_resampled.T.apply(lambda x: x / np.linalg.norm(x)), alpha=0.5)
plt.title('Neural tuning curves (resampled)')
plt.xlabel('Stimulus Frequency (kHz)')
plt.ylabel('Normalized response ($ΔF/F_0$)')
plt.xscale('log', base=2)
plt.tight_layout()
plt.savefig(figspath('tuning-curves-resampled.png'), dpi=300)
plt.show()
tuning_curves_resampled